# 概述

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

本章回答一個系統性問題：**配對交易的績效由哪幾個可分離的層決定，各層貢獻多少？**

透過形成期架構的中性化重構，把策略拆解為三個可獨立替換的層——**特徵 / 分群 / 排序**——
再以宣告式方式展開「3 種分群 × 3 種排序」的消融矩陣，最後在最佳配對底上疊加
**DRL 交易端**與 **regime 進場閘門**兩個正交增益，量化每一層的邊際貢獻。


# 一、形成期架構中性化

## 動機：可解釋的模組邊界

重構前，各分群策略互相 import 對方的內部方法——`Agglomerative` 從 `HDBSCAN` 借特徵、
`HDBSCAN_Cluster` 從 `DTW` 借排序。這種耦合無法向審查者解釋（「為什麼 Agglomerative 要引用 HDBSCAN？
未來 K-means 又該引用誰？」）。

## 三層中性化

把三個共用能力抽成與**演算法無關**的中性模組：

| 中性層 | 職責 | 可插拔選項 |
| :--- | :--- | :--- |
| `_features.py` | 報酬 PCA 因子載荷、因子殘差化、混合特徵（⊕ 基本面 ⊕ 產業） | price_pca / fundamentals_mix |
| `_clustering.py` | 特徵矩陣 → 群標籤 | **hdbscan / agglomerative / kmeans** |
| `_ranking.py` | 群內共整合篩選 + 距離排序 | **ssd / dtw / ssd_dtw_pca** |

策略 = 三層的**組態**，由中性組裝器 `cluster_formation.py` 依參數組裝。
新增分群法（如 K-means）只需在 `_clustering.py` 加一個 backend；新增排序準則同理。

## bit-identical 驗證

抽取過程以數值回歸測試（`tools/formation_regression.py`）保證**逐位元等價**：
組裝器的「Agglomerative × SSD × 混合特徵」組合與重構前的現役策略選出**完全相同的配對**
（雜湊一致，20/20 對），確保架構重構不改變任何已驗證的實證結果。


# 二、參考文獻與方法對應

## 分群方法

> Campello, R. J., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD*.
> Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *JASA*, **58**(301).
> MacQueen, J. (1967). Some methods for classification and analysis of multivariate observations. *Berkeley Symposium*.

- **HDBSCAN**（Campello 2013）：密度聚類，自動決定群數、標記噪音（⚠️ `ref/` 缺 PDF）
- **Agglomerative**（Ward 1963）：階層聚類，dendrogram 分位數校準 distance_threshold（⚠️ 缺 PDF）
- **K-means**（MacQueen 1967）：分割式聚類，需預指定群數——本研究以**同期 Agglomerative 的群數**作為 k（資料驅動、量級可比）（⚠️ 缺 PDF）

## 排序準則

> Gatev et al. (2006) — SSD 距離；許鈞翔 (2025) — DTW 與 SSD-DTW-PCA 融合排序

- **SSD**：標準化價格歐氏距離（同步共動）；**DTW**：Sakoe-Chiba 時間扭曲距離（容許領先/落後）；
  **SSD-DTW-PCA**：兩距離標準化後 PCA 融合取第一主成分
  （📄 `ref/2006-...pdf`、`ref/2025-最小距離法...(許鈞翔).pdf`）

## 特徵與交易端

> Avellaneda & Lee (2010) — 報酬 PCA 因子載荷；Hong & Hwang (2021) — 基本面配對；
> Kim & Kim (2019) — DRL 門檻選擇式交易

混合特徵（報酬 PCA ⊕ log 市值/盈餘殖利率 ⊕ GICS one-hot）為所有 9 格的**固定控制變因**，
基本面取自 FMP Point-in-Time（無前視）。DRL 交易端見 `trading/drl_threshold_trading.ipynb`。


# 三、3×3 分群 × 排序消融矩陣

**實驗設計**：固定混合特徵、固定 Z-Score 交易端（控制變因），唯一變因 = 分群方法 × 排序準則。
9 格全部由中性組裝器宣告式展開，K-means 群數對齊同期 Agglomerative。

## 最佳年化報酬（%）／最佳 Sharpe

| 分群＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **HDBSCAN** | 1.29 / 0.17 | 0.89 / 0.15 | **1.70 / 0.23** |
| **Agglomerative** | **1.77 / 0.26** | 0.60 / 0.11 | 1.13 / 0.18 |
| **K-means** | 1.31 / 0.20 | −0.06 / 0.03 | 0.67 / 0.12 |

> 全部 Top 1；Agglomerative × SSD 格經回歸測試確認 bit-identical 復現現役 Agglomerative (FMP) 1.77%。


## 矩陣的三個發現

**1. 全域最佳 = Agglomerative × SSD（1.77% / Sharpe 0.26）**
混合特徵（基本面 + 產業）的分群傾向聚合「同步共動」股票，與同步性度量（SSD）最相容。

**2. 排序準則的優劣依賴分群方法（交互作用，非單純主效應）**

- Agglomerative / K-means 下：**SSD > SSD-DTW-PCA > DTW**（單調遞減）
- HDBSCAN 下**反轉**：**SSD-DTW-PCA (1.70%) > SSD (1.29%) > DTW**

HDBSCAN × SSD-DTW-PCA 是全矩陣第二高——**只有做完整 3×3 才看得到的意外組合**；
沿對角線試（每分群配其「自然」排序）會錯過它。

**3. DTW 排序在混合特徵下全面偏弱**（三種分群下皆最差，K-means×DTW 甚至轉負）
時間扭曲度量與「基本面/產業相近但價格路徑未必 lead-lag」的分組不相容；
K-means 的等向球狀假設再放大此錯配。

**分群方法本身**（沿各自最佳排序）：Agglomerative (1.77) ≈ HDBSCAN (1.70) > K-means (1.31)——
K-means 最弱，符合「等向球狀假設不適合含 one-hot 的混合特徵空間」。


# 四、DRL 交易端疊加（正交第二層）

取矩陣 top-2 配對底（AGG-SSD、HDB-SDP），**固定配對、只把 Z-Score 換成 DRL 門檻選擇式交易端**
（借用已算好的形成期配對，零重跑）。

| 格 | Z-Score（矩陣值） | → DRL | Δ年化 |
| :--- | :---: | :---: | :---: |
| **AGG-SSD** | 1.77% / Sh 0.26 / PF 1.20 | **2.65% / 0.38 / PF 1.46** | **+0.88pp** |
| **HDB-SDP** | 1.70% / 0.23 / PF 1.19 | **2.43% / 0.31 / PF 1.37** | +0.73pp |

- 四項指標（年化、Sharpe、PF、正Sharpe格數 3–4→8）全面提升，MDD 不惡化
- DRL 增益隨配對品質放大：本處混合特徵底的 +0.7~0.9pp，大於早期純基本面底的 +0.4pp
- HDB-SDP 的亮點延續到 DRL 層——證明它不是 Z-Score 層的偶然，底層配對訊號紮實


# 五、三層疊加：配對 × DRL × regime 閘門（五輪變異數）

在 DRL 之上再疊 **DG25 低分散度進場閘門**（分散度處於歷史最低四分位時暫停新開倉，
walk-forward 無前視）。DRL 未固定隨機種子，故以**五輪獨立重訓**取正式口徑。

| 三層策略 | 最佳年化 中位［範圍］ | 最佳 Sharpe 中位［範圍］ | 正Sharpe最少 |
| :--- | :---: | :---: | :---: |
| **AGG-SSD-DRL-DG25** | **2.69%**［2.65, 2.71］ | **0.40**［0.39, 0.40］ | **14/15** |
| **HDB-SDP-DRL-DG25** | 2.71%［2.61, 2.83］ | 0.35［0.34, 0.36］ | 7/15 |

**閘門的邊際貢獻**（vs 兩層 DRL）：

- **AGG-SSD**：年化中性（2.65→2.69），但**全網格轉正（8→14–15/15）**、MDD −23.5→−21.9、PF 1.46→1.49
  ——閘門的價值在**穩健性**（避開平靜期無溢酬交易），而非年化
- **HDB-SDP**：年化 **+0.28pp（2.43→2.71）**、MDD −26.5→−20.9、PF 1.37→1.59——三項全面改善

> **AGG-SSD-DRL-DG25 是全專案最穩健的策略**：五輪年化範圍僅 0.06pp、Sharpe 0.39–0.40、最差輪仍 14/15 正 Sharpe。


# 七、第三維度：共整合篩選的貢獻

形成期的第三個可消融維度——**篩選**（ADF 共整合 + OU 半衰期 + Hurst 三道統計過濾）。
中性化後（`_cointegration.screen_pair`）篩選成為可開關參數，得以量化其邊際貢獻。

## 實驗設計：三項對照

| 實驗 | 分組 | 排序 | 篩選 |
| :--- | :--- | :--- | :---: |
| ① 同產業純排序 | GICS | SSD / DTW / SDP | ✗ |
| ② 同產業排序 + 篩選 | GICS | SSD / DTW / SDP | ✓ |
| ③ 分群 + 排序 + 篩選 | HDB / AGG / KM | SSD / DTW / SDP | ✓ |

## 篩選的貢獻（GICS 分組固定，最佳年化）

| 排序 | 無篩選 | 有篩選 | Δ年化 |
| :--- | :---: | :---: | :---: |
| SSD | 0.79% / Sh 0.22 | **1.66% / 0.20** | **+0.87pp** |
| DTW | 0.86% / 0.20 | 1.10% / 0.18 | +0.25pp |
| SSD-DTW-PCA | 1.00% / 0.21 | **1.65% / 0.35** | +0.65pp |

**三道統計過濾是真實且一致的增益來源**——三種排序準則下全部為正，SSD 下貢獻近乎翻倍。
純距離排序不足以識別可交易配對：距離度量只回答「歷史走勢多接近」，
共整合檢定才回答「價差是否真的會回歸」，兩者結合才構成有效的配對選取。


# 八、四種分組 × 三種排序（皆含篩選）

加入 GICS 產業分組後的完整對照——**傳統產業分組 vs 三種資料驅動分群**：

| 分組＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **GICS（傳統）** | 1.66% / 0.20 | 1.10% / 0.18 | 1.65% / **0.35** |
| **HDBSCAN** | 1.29% / 0.17 | 0.89% / 0.15 | **1.70%** / 0.23 |
| **Agglomerative** | **1.77%** / 0.26 | 0.60% / 0.11 | 1.13% / 0.18 |
| **K-means** | 1.31% / 0.20 | −0.06% / 0.03 | 0.67% / 0.12 |

## 誠實的負面結果：ML 分群相對 GICS 的優勢有限

- 最佳 ML 分群（Agglomerative × SSD，1.77%）僅**高於 GICS × SSD（1.66%）0.11pp**
- HDBSCAN 與 K-means 在 SSD 排序下**反而輸給傳統產業分組**
- **GICS × SSD-DTW-PCA 是全表最高 Sharpe（0.35）**，年化 1.65% 與最佳者相當

即：在目前的特徵設定下，「資料驅動分群優於產業分類」這個常見假設**未獲支持**。
這對論文是有價值的誠實結果，也指向下一步的關鍵問題——分群品質受限於**特徵**（見下節）。


## 對照文獻：特徵維度的落差

ref 內的 ML 配對交易文獻使用的特徵集遠比本研究豐富：

> Sanders (2021), *Pairs Trading via Unsupervised Learning*：
> 特徵集 = **48 個動量因子**（1–48 月累積報酬）+ **78 個公司特徵**（取自 Green et al. 2017），
> 每月更新，以 CRSP 全市場為樣本。作者明言公司特徵「更具前瞻性」——
> 走勢曾同步的兩檔股票，若公司特徵迥異，未來可能分道揚鑣。

本研究目前的分群特徵為 **5 維報酬 PCA 因子載荷 ⊕ 2 維基本面（log 市值、盈餘殖利率）
⊕ 12 維 GICS one-hot**，維度與資訊量都遠低於上述文獻。這很可能是 ML 分群未能顯著
超越 GICS 的主因：**分群演算法只能在給定特徵空間中尋找結構，特徵不足時再好的
演算法也無法區分出有意義的群落**。

後續方向見專案規劃：擴充動量因子集、加入更多可取得的公司特徵與流動性/波動特徵。


# 六、完整策略地圖：每一層的邊際貢獻

```
                配對品質        + DRL 交易端      + regime 閘門
                (Z-Score)      (門檻選擇式)      (DG25，五輪中位)
AGG-SSD  ──────  1.77%  ────────►  2.65%  ────────►  2.69%
                (Sh 0.26)         (+0.88pp)          (全網格轉正、MDD↓、PF↑)
HDB-SDP  ──────  1.70%  ────────►  2.43%  ────────►  2.71%
                (Sh 0.23)         (+0.73pp)          (+0.28pp、MDD↓、PF↑)
```

## 三層各自的性質（研究結論）

| 層 | 增益來源 | 性質 |
| :--- | :--- | :--- |
| **配對品質**（分群×排序） | 選出真正具均衡關係的配對 | 決定天花板；混合特徵 + 同步性排序最佳 |
| **DRL 交易端** | 每配對自選進出場門檻 + 拒絕壞配對 | 對及格配對底穩定 +0.7~0.9pp，隨配對品質放大 |
| **regime 閘門** | 只在分散度高（有套利溢酬）時進場 | 主要提升**穩健性**（全網格轉正、MDD↓、PF↑），年化增益視配對底而定 |

三層皆為**正交**（各自的消融都獨立驗證有效），可疊加。這是本研究對「配對交易績效可分解性」的核心貢獻。

## 已知限制

- DRL 為五輪重訓口徑，單次數字有 ±0.1 Sharpe 級別波動；跨策略比較以中位數±範圍為準
- 混合特徵的基本面為 FMP 月頻 PIT，部分已下市股票以產業中位數插補
- regime 閘門的分散度分位在極端轉折（如 2020/03）有約一個月辨識落後
- 9 格矩陣固定 Top 1 為最佳；其餘 Top N 的完整網格見 `comparison.ipynb`
